# ITS_LIVE velocities over the Shisper tongue and ice dam

Reads image-pair velocities from the ITS_LIVE version 2 datacube (120 m) for 74.54 to 74.63° E and 36.34 to 36.41° N, directly from its public AWS archive (no login). It keeps every pixel of the cube's glacier mask and a 1.2 km margin around it, because the mask predates the 2018 to 2019 advance and the margin contains the ice dam; the `landice` column marks the pixels inside the mask.

**Output** (`Shisper_velocity_ITSLIVE`): `shisper_velocity.csv.gz` (one row per pixel and image pair: `lat`, `lon`, `mid_date`, `v [m/yr]`, `vx [m/yr]`, `vy [m/yr]`, `v_error [m/yr]`, `date_dt [days]`, `satellite_img1`, `satellite_img2`, `landice`), `velocity_check.png` and `info.json`. CPU runtime, 15 to 40 minutes.

## Setup

In [ ]:
%pip install -q zarr s3fs

import os, re, json, time, gzip, warnings
import numpy as np, pandas as pd, xarray as xr, requests
from pyproj import Transformer
from shapely.geometry import shape, Point
from scipy import ndimage
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/Shisper_velocity_ITSLIVE'
os.makedirs(OUT, exist_ok=True)

BOX = dict(west=74.54, south=36.34, east=74.63, north=36.41)
BUFFER_PX = 10                                    # margin of 10 x 120 m around the glacier mask
CATALOG = 'https://its-live-data.s3.amazonaws.com/datacubes/catalog_v02.json'
OUTBURSTS = ['2019-06-22', '2020-05-29', '2021-05-15', '2022-05-07']   # Lovell and Muhammad (2025)
LAKE = (36.36061, 74.56276)

## Open the datacube

In [ ]:
cat = requests.get(CATALOG, timeout=300).json()
centre = Point((BOX['west'] + BOX['east']) / 2, (BOX['south'] + BOX['north']) / 2)
ZONE = 32600 + int((centre.x + 180) // 6) + 1

def epsg_of(props):
    m = re.search(r'EPSG:?(\d{4,5})', str(props.get('zarr_url', '')))
    if m:
        return int(m.group(1))
    for k in ('epsg', 'data_epsg', 'projection'):
        m = re.search(r'(\d{4,5})', str(props.get(k, '')))
        if m:
            return int(m.group(1))
    return None

feats = [f for f in cat['features'] if f['properties'].get('zarr_url') and f['properties'].get('datacube_exist', 1)
         and shape(f['geometry']).contains(centre)]
assert feats, 'no ITS_LIVE datacube covers this box'
feats.sort(key=lambda f: epsg_of(f['properties']) != ZONE)     # prefer the cube in the box's own UTM zone
props = feats[0]['properties']
URL, EPSG = props['zarr_url'], epsg_of(props)
key = re.sub(r'^(s3://its-live-data/|https?://its-live-data\.s3[.a-z0-9-]*\.amazonaws\.com/)', '', URL)
S3URL, HTTPURL = f's3://its-live-data/{key}', f'https://its-live-data.s3.amazonaws.com/{key}'
try:
    ds = xr.open_zarr(S3URL, storage_options={'anon': True}, consolidated=True)
except Exception:
    ds = xr.open_zarr(HTTPURL, consolidated=True)
if not EPSG:
    src = ds.attrs.get('projection') or (ds['mapping'].attrs.get('spatial_epsg') if 'mapping' in ds else '')
    EPSG = int(re.search(r'(\d{4,5})', str(src)).group(1))
print('datacube:', HTTPURL, '| EPSG', EPSG, '| image pairs:', f"{ds.sizes['mid_date']:,}")

to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{EPSG}', always_xy=True)
to_ll = Transformer.from_crs(f'EPSG:{EPSG}', 'EPSG:4326', always_xy=True)
bx, by = to_utm.transform([BOX['west'], BOX['east'], BOX['west'], BOX['east']],
                          [BOX['south'], BOX['south'], BOX['north'], BOX['north']])
ydesc = bool(ds.y.values[0] > ds.y.values[-1])
W = ds.sel(x=slice(min(bx), max(bx)), y=slice(max(by), min(by)) if ydesc else slice(min(by), max(by)))
ny, nx = W.sizes['y'], W.sizes['x']
assert ny > 0 and nx > 0, 'the box is outside this datacube'

ice = (np.nan_to_num(W['landice'].values.astype('float32')) == 1) if 'landice' in W else np.zeros((ny, nx), bool)
pick = ndimage.binary_dilation(ice, iterations=BUFFER_PX) if ice.any() else np.ones((ny, nx), bool)
XX, YY = np.meshgrid(W.x.values, W.y.values)
LON, LAT = to_ll.transform(XX, YY)

# read in blocks aligned with the cube's chunks
ch = W['v'].encoding.get('chunks') or (None, 10, 10)
BY, BX = int(ch[-2] or 10), int(ch[-1] or 10)
iy0 = int(np.nonzero(ds.y.values == W.y.values[0])[0][0])
ix0 = int(np.nonzero(ds.x.values == W.x.values[0])[0][0])
def edges(n, off, b):
    first = (b - off % b) % b or b
    return [0] + list(range(first, n, b)) + [n]
YE, XE = edges(ny, iy0, BY), edges(nx, ix0, BX)
blocks = [(a, b, c, d) for a, b in zip(YE[:-1], YE[1:]) for c, d in zip(XE[:-1], XE[1:]) if pick[a:b, c:d].any()]
print(f'window {ny} x {nx} | glacier pixels {int(ice.sum())} | pixels extracted {int(pick.sum())} '
      f'({int((pick & ~ice).sum())} in the margin) | blocks {len(blocks)}')

## Extract the time series

In [ ]:
VARS = [v for v in ('v', 'vx', 'vy', 'v_error') if v in W]
NAMES = {'v': 'v [m/yr]', 'vx': 'vx [m/yr]', 'vy': 'vy [m/yr]', 'v_error': 'v_error [m/yr]'}
MD = pd.DatetimeIndex(pd.to_datetime(W['mid_date'].values))
PER_DATE = {}
if 'date_dt' in W:
    dt = W['date_dt'].values
    dt = dt / np.timedelta64(1, 'D') if np.issubdtype(dt.dtype, np.timedelta64) else dt.astype('float64')
    PER_DATE['date_dt [days]'] = dt.astype('float32').round(2)
for s in ('satellite_img1', 'satellite_img2'):
    if s in W:
        a = np.asarray(W[s].values)
        PER_DATE[s] = (np.char.decode(a, 'utf-8') if a.dtype.kind == 'S' else a).astype(object)

# for the check figure: median speed during the surge, and monthly speed histograms of the glacier pixels
SURGE = np.asarray((MD >= '2017-06-01') & (MD <= '2019-06-30'))
SHORT = PER_DATE['date_dt [days]'] <= 120 if 'date_dt [days]' in PER_DATE else np.ones(len(MD), bool)
MON = np.clip(np.asarray((MD.year - 1980) * 12 + MD.month - 1), 0, None)
NM, BIN, NB = int(MON.max()) + 1, 5.0, 2400
H = np.zeros(NM * NB, 'int64')
med_surge = np.full((ny, nx), np.nan, 'float32')

path = f'{OUT}/shisper_velocity.csv.gz'
rows, header, t0 = 0, True, time.time()
with gzip.open(path + '.part', 'wt', compresslevel=3, newline='') as fh:
    for k, (a, b, c, d) in enumerate(blocks):
        for attempt in range(4):
            try:
                sub = W[VARS].isel(y=slice(a, b), x=slice(c, d)).load()
                break
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(15 * (attempt + 1))
        v = sub['v'].values                                                  # (time, y, x)
        if SURGE.any():
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)
                med_surge[a:b, c:d] = np.nanmedian(v[SURGE], axis=0)
        pi, pj = np.nonzero(pick[a:b, c:d])
        vv = v[:, pi, pj]
        t_idx, p_idx = np.nonzero(~np.isnan(vv))
        if len(t_idx):
            gi, gj = a + pi[p_idx], c + pj[p_idx]
            cols = {'lat': LAT[gi, gj].round(6), 'lon': LON[gi, gj].round(6), 'mid_date': MD.values[t_idx]}
            for name in VARS:
                cols[NAMES[name]] = sub[name].values[:, pi, pj][t_idx, p_idx].astype('float32')
            for name, arr in PER_DATE.items():
                cols[name] = arr[t_idx]
            cols['landice'] = ice[gi, gj].astype('uint8')
            pd.DataFrame(cols).to_csv(fh, index=False, header=header)
            header = False
            rows += len(t_idx)
            g = ice[gi, gj] & SHORT[t_idx]
            if g.any():
                bins = np.clip((vv[t_idx[g], p_idx[g]] / BIN).astype(int), 0, NB - 1)
                H += np.bincount(MON[t_idx[g]] * NB + bins, minlength=NM * NB)
        if k % 5 == 0 or k == len(blocks) - 1:
            print(f'  block {k + 1}/{len(blocks)} | rows {rows:,} | {(time.time() - t0) / 60:.1f} min', flush=True)
os.replace(path + '.part', path)
print(f'{rows:,} rows written to {path}')

## Check

In [ ]:
Hm = H.reshape(NM, NB)
months = pd.date_range('1980-01-01', periods=NM, freq='MS')
def q_hist(c, q):
    n = c.sum()
    return (np.searchsorted(np.cumsum(c), q * n) + 0.5) * BIN if n else np.nan
med_m = np.array([q_hist(c, .5) for c in Hm])
ok = Hm.sum(1) >= 20

fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 1.25]})
ext = [W.x.values.min(), W.x.values.max(), W.y.values.min(), W.y.values.max()]
im = ax[0].imshow(med_surge if ydesc else med_surge[::-1], extent=ext, cmap='viridis',
                  vmax=np.nanpercentile(med_surge, 98) if np.isfinite(med_surge).any() else 1)
plt.colorbar(im, ax=ax[0], label='median speed, Jun 2017 to Jun 2019 (m/yr)')
ax[0].contour(XX, YY, ice.astype(float), levels=[0.5], colors='white', linewidths=0.8)
ax[0].plot(XX[pick & ~ice], YY[pick & ~ice], '.', ms=1.5, c='orange', alpha=.7)
lx, ly = to_utm.transform(LAKE[1], LAKE[0])
ax[0].plot(lx, ly, marker='*', ms=16, c='gold', mec='k', ls='')
ax[0].ticklabel_format(useOffset=False, style='plain')
ax[0].set_title('white: glacier mask, orange: margin pixels, star: lake', fontsize=10)
ax[1].plot(months[ok], med_m[ok], '.-', ms=3, lw=.6)
for d in OUTBURSTS:
    ax[1].axvline(pd.Timestamp(d), c='r', lw=.8, ls='--')
ax[1].set_xlim(pd.Timestamp('2013-01-01'), months[-1] + pd.Timedelta(days=31))
ax[1].set_ylabel('monthly median speed, glacier pixels (m/yr)')
ax[1].set_title('image pairs up to 120 days apart; dashed: outbursts', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT}/velocity_check.png', dpi=120); plt.show()

info = dict(datacube=HTTPURL, epsg=EPSG, box=BOX, buffer_px=BUFFER_PX, window_pixels=[int(ny), int(nx)],
            glacier_pixels=int(ice.sum()), extracted_pixels=int(pick.sum()), margin_pixels=int((pick & ~ice).sum()),
            rows=int(rows), first_pair=str(MD.min().date()), last_pair=str(MD.max().date()),
            columns=['lat', 'lon', 'mid_date'] + [NAMES[v] for v in VARS] + list(PER_DATE) + ['landice'])
json.dump(info, open(f'{OUT}/info.json', 'w'), indent=2)
print(json.dumps(info, indent=1))